In [1]:
import base64
import json
import random

import pandas as pd
import requests
import tensorflow as tf

In [2]:
df = pd.read_csv("data/customer.csv")
y = df.pop("Response")

columns = df.columns.values
rand = random.randint(0, len(df))

features = df.values[rand]
label = y[rand]

inputs = {key: value for key, value in zip(columns, features)}
inputs


{'Unnamed: 0': 236.0,
 'Education': 2.0,
 'Marital_Status': 0.0,
 'Income': 23661000.0,
 'Kidhome': 1.0,
 'Teenhome': 0.0,
 'Recency': 46.0,
 'MntCoke': 18000.0,
 'MntFruits': 0.0,
 'MntMeatProducts': 4000.0,
 'MntFishProducts': 0.0,
 'MntSweetProducts': 0.0,
 'MntGoldProds': 1000.0,
 'NumDealsPurchases': 1.0,
 'NumWebPurchases': 1.0,
 'NumCatalogPurchases': 0.0,
 'NumStorePurchases': 3.0,
 'NumWebVisitsMonth': 7.0,
 'AcceptedCmp3': 0.0,
 'AcceptedCmp4': 0.0,
 'AcceptedCmp5': 0.0,
 'AcceptedCmp1': 0.0,
 'AcceptedCmp2': 0.0,
 'Complain': 0.0,
 'Z_CostContact': 3.0,
 'Z_Revenue': 11.0}

In [3]:
def string_feature(value):
    return tf.train.Feature(
        bytes_list=tf.train.BytesList(value=[bytes(value, "utf-8")]),
    )


def float_feature(value):
    return tf.train.Feature(
        float_list=tf.train.FloatList(value=[value]),
    )


def int_feature(value):
    return tf.train.Feature(
        int64_list=tf.train.Int64List(value=[value]),
    )


In [4]:
def prepare_json(inputs: dict):
    feature_spec = dict()

    for keys, values in inputs.items():
        if keys == "Income":
            feature_spec[keys] = float_feature(float(values))
        else:
            feature_spec[keys] = int_feature(int(values))

    example = tf.train.Example(
        features=tf.train.Features(feature=feature_spec)
    ).SerializeToString()

    result = [{"examples": {"b64": base64.b64encode(example).decode()}}]

    return json.dumps(
        {
            "signature_name": "serving_default",
            "instances": result,
        }
    )

In [5]:
def make_predictions(inputs):
    json_data = prepare_json(inputs)

    endpoint = (
        "https://mlops-final-proyek-production.up.railway.app/v1/models/customerpersonality-model:predict"
    )
    response = requests.post(endpoint, data=json_data)
    prediction = response.json()["predictions"][0][0]

    if prediction < 0.6:
        return "Tidak berpartisipasi"
    else:
        return "Berpartisipasi"

In [6]:
make_predictions(inputs)

'Tidak berpartisipasi'

In [7]:
label

0